# EABL Hierarchical Demand Forecasting Pipeline

## The problem this project solves

**"How much will we sell next month?"** is really several different questions stacked on top of each other: how much of *this exact SKU* (e.g. Tusker Lager 500ml), how much of *this brand* overall, how much of *this category* (Beer, Spirits, RTD...), and how much *company-wide* — each of those, further split by sales channel (Bar, Wholesaler, Nightclub, etc.).

A single company-wide forecast can be perfectly accurate at the top while being badly wrong underneath — overstocked on a Spirits SKU nobody wants, understocked on the Tusker everyone's asking for — because opposite-signed errors cancel out in the total. That's expensive both ways: stockouts lose sales, overstock ties up cash and risks spoilage.

**This pipeline forecasts at every level at once** — SKU, Brand, Category2, and channel segment, all the way up to the company total — and mathematically guarantees they agree with each other (SKU forecasts sum to Brand forecasts, which sum to Category forecasts, which sum to the Total). That's what "hierarchical" means here: not one model, but a coherent set of forecasts across the whole product/channel structure, so a good-looking top-line number can never hide a broken picture underneath.

## How this project is conducted — the logic, end to end

1. **Build one clean fact table** from two raw sources (`eabl_sales_consolidated_cleaned.csv` and `product_master_cleaned.csv`), fixing real, confirmed data problems along the way: a product-master file where descriptions aren't actually unique keys, a naming mismatch between the two systems for one product, and inconsistent measurement units (some products counted in bottles, others already in cases).
2. **Screen for data-quality issues** (duplicate-looking rows, statistical outliers, negative quantities, discontinued SKUs still selling) — investigate each one rather than blindly dropping rows.
3. **Define the hierarchy explicitly** — SKU → Brand → Category2 → Total, crossed with channel segment — using `hierarchicalforecast`'s `aggregate()`, which builds the mathematical structure (a "summing matrix") that later guarantees coherence.
4. **Resample to weekly and fill gaps** — daily data is too sparse for reliable forecasting at the SKU×segment level; weekly is the right grain for a stocking decision.
5. **Identify "cold-start" series** — SKU×segment combinations with too little history to trust a model built on lag features, so they can be handled differently.
6. **Engineer features specific to this business**: recent sales history (lags, rolling averages), Kenyan public holidays, payday timing (salaries land late-month in Kenya — a real driver of alcohol demand), and the product hierarchy itself as a feature, so one global model can learn patterns across products instead of being trained in isolation per SKU.
7. **Validate the core assumption before trusting it** — check directly in the data whether different categories/channels actually spike by different amounts around holidays, rather than just asserting it.
8. **Fit base forecasts** — a global LightGBM model as the primary engine, plus simple per-series baselines (AutoETS, SeasonalNaive) as an honesty check.
9. **Reconcile** — force every level of the hierarchy to agree mathematically. This is the step that actually makes it "hierarchical," not just "forecast a bunch of related series."
10. **Evaluate honestly** — using rolling-origin backtesting (not a single lucky train/test split) and a scale-normalized metric (WMAPE), and *act* on what it says — including the uncomfortable finding that the tuned model didn't automatically beat a simple baseline everywhere.
11. **Diagnose and fix what the evaluation surfaced** — cold-start series were forecast badly by every model tried, so they're routed to a Brand-level share estimate instead, rather than the finding being noted and ignored.

## Packages used, and what each one actually does here

| Package | Role in this pipeline |
|---|---|
| **pandas** | The data backbone — every table (`fact`, `Y_df`, `S_df`, `reconciled`) is a pandas DataFrame. |
| **pathlib** | Just used to create the output directory (`data/processed/`) before writing files into it. |
| **hierarchicalforecast** (`aggregate`, `HierarchicalReconciliation`, `BottomUp`, `MinTrace`) | The library that actually makes this "hierarchical" — `aggregate()` builds the multi-level structure and summing matrix; `HierarchicalReconciliation` + `MinTrace`/`BottomUp` are what force every level to mathematically agree. |
| **holidays** | Supplies a real Kenyan public-holiday calendar (`holidays.Kenya(...)`) — used to build the `is_holiday_week` feature, instead of hand-listing dates. |
| **mlforecast** (`MLForecast`, lag/rolling-window transforms) | Wraps a plain ML model (LightGBM) into a proper multi-series forecaster — auto-generates lag and rolling-window features per series, handles the recursive multi-step prediction, and exposes rolling-origin backtesting (`cross_validation`). |
| **statsmodels** (`STL`) | Seasonal-Trend decomposition — used once, specifically to check whether Beer and Spirits actually have different holiday-season spikes in the real data, before trusting that assumption. |
| **lightgbm** | The actual gradient-boosted-tree model doing the forecasting — chosen because it can learn shared patterns across hundreds of series at once when given the right features (lags, calendar effects, category identity). |
| **statsforecast** (`AutoETS`, `SeasonalNaive`) | Fast classical statistical forecasting models, fit independently per series — used as a sanity-check baseline. If a complex tuned model can't beat these, that's a real signal, not a detail to ignore (and in this project, it *was* a real signal for cold-start series). |
| **utilsforecast** (`evaluate`, `mase`) | Evaluation utilities built for multi-series forecasts — supplies metrics like MASE in a way that works across hundreds of series and hierarchy levels at once. |
| **optuna** | Hyperparameter search — used to tune the LightGBM model's settings (learning rate, tree depth, etc.) against the actual backtest score, rather than guessing them. |
| **functools.partial** | Plain Python utility — used to pre-fill the `seasonality` argument into `mase` before handing it to the evaluator, since the evaluator only calls the metric with `(df, models, train_df, ...)`. |

## A note on how to read the rest of this notebook

Every major stage below has its own markdown cell immediately before it, explaining what that group of code cells does and why, cell by cell. Sanity-check numbers you should expect to see (row counts, percentages) are called out explicitly, so you can tell at a glance whether a rerun produced the same shape of result.


In [689]:
import pandas as pd
from pathlib import Path
from hierarchicalforecast.utils import aggregate
import holidays
from mlforecast.lag_transforms import RollingMean, RollingStd
from statsmodels.tsa.seasonal import STL
import lightgbm as lgb
from mlforecast import MLForecast
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace
from functools import partial
from utilsforecast.losses import mase
from utilsforecast.evaluation import evaluate
from mlforecast.utils import PredictionIntervals
import optuna


## Step 01 — Build the unified fact table

**Goal of this whole stage:** turn two raw files — a transaction-level sales log and a product master — into one clean table (`fact`) where every sale has the correct product hierarchy attached, in a consistent unit, with no missing values. Every later step depends on this being right, so this stage does more defensive checking than a typical "load and join."

**Cell-by-cell:**

- **Load & setup** — creates the output folder (`to_parquet`/`to_csv` won't create missing directories on their own) and loads both raw CSVs, parsing `visit_date` as a real datetime immediately so every later date operation works correctly.
- **`products` (display)** — a raw look at the product master as loaded: 881 rows, 28 columns, one row per product code.
- **`products_by_description`** — a critical fix, not a formatting step: `Product_Description` is **not actually a unique key** in the product master (7 descriptions map to two different `Product_Code`s each — a legacy code reissue, confirmed by direct inspection). This sorts so the `Active` status and the newer code win, then deduplicates — without this, the merge below would either error or silently double-count sales.
- **`products_by_description` (display)** — confirms the dedup worked: 874 rows, down from 881.
- **Unmatched-rows diagnostic (`fact_before`)** — merges sales to products *without* the alias fix below, specifically to measure the "before" state: 1,223 unmatched rows, 730 of them being one specific product string (`"Smirnoff Ice Raspberry Twist 300ml in Bottle"`). This cell exists purely to produce a comparable baseline for the next cell's fix.
- **Alias fix + real merge** — the sales system and the product master name one real product differently ("in Bottle" vs. "RET", confirmed to be the same 300ml SKU by cross-checking EABL's own naming convention on 108 other products). This renames it before merging, recovering those 730 real transactions instead of losing them. `fact.isna().sum()` afterward shows exactly 493 rows still unmatched — down from 1,223.
- **Assert + drop unmatched** — a tripwire (`assert unmatched_pct < 0.02`) that stops the pipeline if a future data refresh breaks the join badly, followed by dropping whatever's still unmatched. Those 493 rows were confirmed to be genuinely outside the EABL SKU hierarchy — third-party mixers, sample giveaways, expired write-offs, promotional merchandise — not real product demand. This drop is also what makes `Category2`/`Brand` null-free, which the hierarchy-building step later requires.
- **`fact` (display)** — the joined table at this point: 984,292 rows, 17 columns.
- **Unit normalization + finalize** — converts `quantity`/`return_quantity` into a consistent unit (`quantity_cases`) using each product's own `Cases_Conversion` factor (some products are recorded per-bottle, others already per-case — a single blanket conversion would be wrong). Also fixes a second real bug found later: the raw sales file has *two* differently-named segment columns (`segment` and `segment_grouped`); the old code's rename created a silent duplicate-column collision, so this drops the raw `segment` first. Saves the result as `data/processed/fact_table.csv`.
- **`fact.dtypes`** — a quick sanity check that `ds` is a real `datetime64`, not a string — this matters because reading the CSV back later requires `parse_dates=["ds"]` explicitly, since CSV doesn't preserve dtypes the way Parquet would.


In [690]:
Path("ForecastData/ForecastDataProcessed").mkdir(parents=True, exist_ok=True)   # to_parquet does NOT create missing dirs

sales = pd.read_csv("../eabl parsed csv/eabl_sales_consolidated_cleaned.csv",parse_dates=["visit_date"])
products = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")



In [691]:
products

,Product_Code,Product_Description,Handheld_Description,Status,Default_UOM,Category,Sub_Category,Sub_Category2,New_Sub_Category2,Brand,...,Sub_Category1,CATMAN_Category,FY25_July_NSV_Conversion_1st,FY25_November_NSV_Conversion_1st,FY25_January_NSV_Conversion_1st,FY26_October_NSV_Conversion_18th,FY25_EX_KBL_Price_Per_Case,FY26_October_NSV_Conversion_18th_Updated,FY26_June_NSV_Conversion_15th,FY26_EX_KBL_Price_Per_Case
0,47770,Cameron Brig 70Cl 9Y 12X01,Cameron Brig 9Y 70cl,Inactive,BTL,Spirits,Premium,Premium Core,Premium Core,Cameron Brig,...,Whisky - Single Grain Scotch,1,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,0.000000
1,48453,Pimms 75Cl 12X01,Pimms 750ml,Active,BTL,Spirits,Premium,Premium Core,Premium Plus,Pimms,...,Liqueur,Cane/Bitters,1182.067032,1182.067032,1261.884957,1261.884957,20438.14000,1261.884957,1261.884957,20175.438596
2,207134,Dalwhinnie 75cl 15Y 06X01*,Dalwhinnie 75cl*,Active,BTL,Spirits,Reserve,Reserve,Reserve,Dalwhinnie,...,Whisky - Single Malt Scotch,Whisky,4963.071266,4963.071266,4907.889191,4907.889191,35681.51079,4907.889191,4907.889191,36403.508772
3,207145,Cragganmore 12Y 75Cl 12Y 06X01,Cragganmore 12Y 75cl,Inactive,BTL,Spirits,Reserve,Reserve,Reserve,Cragganmore,...,Whisky - Single Malt Scotch,Whisky,4080.836945,4080.836945,4048.154870,4048.154870,29475.37054,4048.154870,4048.154870,30263.157895
4,207148,Cragganmore 12Y 1L 12Y 06X01,Cragganmore 12Y 1L,Inactive,BTL,Spirits,Reserve,Reserve,Reserve,Cragganmore,...,Whisky - Single Malt Scotch,Whisky,5252.065715,5252.065715,5208.489615,5208.489615,39035.08594,5208.489615,5208.489615,39035.087719
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876,FB681232,Free Balozi Lager 500ml Per Bottle,Free Balozi Lager 500ml Per Bottle,Active,BTL,Beer,Mainstream,Mainstream Beer,Mainstream Beer,Balozi Lager,...,1,1,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
877,FB726668,Free Tusker Cider 500ML Per Bottle,Free Tusker Cider 500ML Per Bottle,Active,BTL,Beer,Mainstream,Mainstream Beer,Mainstream Beer,Tusker Cider,...,Cider/ Perry,Lager,0.000000,0.000000,0.000000,0.000000,0.00000,3275.404106,3275.404106,0.000000
878,FB778513,Free Smirnf Ice PP R 300ml Per Bottle,Free S.Ice Pineapple Punch 300ml,Active,BTL,Spirits,RTD,RTD,RTD,Smirnoff Ice Pineapple Punch,...,Ready To Drink,RTD,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
879,FB786887,Free Pilsner Mfalme 300ml RET 25X01 7% In Bott...,Free Pilsner Mfalme 300ml in Bottle,Active,BTL,Beer,Mainstream,Mainstream Beer,Mainstream Beer,Pilsner Mfalme,...,Lager,Lager,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000


In [692]:
# Product_Description is NOT unique in product_master_cleaned.csv — 7 descriptions
# map to 2 Product_Codes each (legacy code reissue, confirmed by inspection).
# 'Active' sorts before 'Inactive' alphabetically; higher Product_Code = more recent.
products_by_description = (
    products.sort_values(["Product_Description", "Status", "Product_Code"],
                        ascending=[True, True, False])
            .drop_duplicates(subset="Product_Description", keep="first")
            .reset_index(drop=True)
)


In [693]:
products_by_description

,Product_Code,Product_Description,Handheld_Description,Status,Default_UOM,Category,Sub_Category,Sub_Category2,New_Sub_Category2,Brand,...,Sub_Category1,CATMAN_Category,FY25_July_NSV_Conversion_1st,FY25_November_NSV_Conversion_1st,FY25_January_NSV_Conversion_1st,FY26_October_NSV_Conversion_18th,FY25_EX_KBL_Price_Per_Case,FY26_October_NSV_Conversion_18th_Updated,FY26_June_NSV_Conversion_15th,FY26_EX_KBL_Price_Per_Case
0,350600,330ml Chic Bottle,Alvaro Bottle (Chic),Active,PC,Returnable,Returnable,Returnable,Returnable,Returnable,...,Bottles/Crates,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
1,350601,500ml Genius Amber,GK Bottle (Genius),Inactive,PC,Returnable,Returnable,Returnable,Returnable,Returnable,...,Bottles/Crates,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
2,616860,Allsopps 500ml 25X01,Allsopps Lager 500ml,Inactive,CAS,Beer,Value,Value,Value,Allsopps,...,Lager,Lager,957.128911,957.128911,1177.742464,1177.742464,3198.850800,1177.742464,1177.742464,3175.678537
3,B716077,Allsopps Lager 500ml Per Bottle,Allsopps Lager 500ml Per Bottl,Active,BTL,Beer,Value,Value,Value,Allsopps,...,Lager,Lager,957.128911,957.128911,1177.742464,1177.742464,3198.850800,1177.742464,1177.742464,3175.678537
4,716077,Allsopps Lager 500ml Ret 25X01 New Look,Allsopps Lager 500ml,Active,CAS,Beer,Value,Value,Value,Allsopps,...,Lager,Lager,957.128911,957.128911,1177.742464,1177.742464,3198.850800,1177.742464,1177.742464,3175.678537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869,636122,Windhoek Lager 330ml Nrb 24X01,Windhoek,Inactive,BTL,Beer,Premium,Premium Beer,Premium Beer,Windhoek Lager,...,Lager,Lager,1.000000,1.000000,1.000000,1.000000,2597.189209,1.000000,1.000000,2597.189209
870,754931,Zacapa Cent 23 75cl 06X01 GX ROW,Zacapa Cent 23 750ml,Active,BTL,Spirits,Reserve,Reserve,Reserve,Zacapa,...,Rum,Rum,5530.221902,5530.221902,5497.539827,5497.539827,40350.878400,5497.539827,5497.539827,40350.877193
871,663823,Zacapa Cent 23 1L 06X01,Zacapa Cent 23 1L,Active,BTL,Spirits,Reserve,Reserve,Reserve,Zacapa,...,Rum,Rum,6827.484146,6827.484146,6783.908046,6783.908046,50000.000000,6783.908046,6783.908046,50000.000000
872,663821,Zacapa Cent 23 75Cl 06X01,Zacapa Cent 23 750ml,Active,BTL,Spirits,Reserve,Reserve,Reserve,Zacapa,...,Rum,Rum,5530.221902,5530.221902,5497.539827,5497.539827,40350.878400,5497.539827,5497.539827,40350.877193


In [694]:
sales_raw = pd.read_csv("../eabl parsed csv/eabl_sales_consolidated_cleaned.csv")
products = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")

products_by_description = (
    products.sort_values(["Product_Description", "Status", "Product_Code"],
                          ascending=[True, True, False])
            .drop_duplicates(subset="Product_Description", keep="first")
)

# merge WITHOUT applying PRODUCT_ALIASES first — this is the "before" state
fact_before = sales_raw.merge(
    products_by_description[["Product_Description", "Product_Code", "Brand", "Category2"]],
    left_on="product", right_on="Product_Description",
    how="left", indicator=True,
)

unmatched_before = fact_before[fact_before["_merge"] == "left_only"]

print("total unmatched rows:", len(unmatched_before))
print("of those, rows for the Smirnoff Ice 'in Bottle' string:",
      (unmatched_before["product"] == "Smirnoff Ice Raspberry Twist 300ml in Bottle").sum())


total unmatched rows: 1223
of those, rows for the Smirnoff Ice 'in Bottle' string: 730


In [695]:
# manual alias for a confirmed naming-convention mismatch: the sales system logs
# this Smirnoff Ice variant as "in Bottle", product master calls it "RET" (returnable) —
# same SKU, same 300ml pack. Recovers 730 rows that would otherwise silently drop below.
PRODUCT_ALIASES = {
    "Smirnoff Ice Raspberry Twist 300ml in Bottle": "Smirnoff Ice Raspberry Twist 300ml RET 25X01",
}
sales["product"] = sales["product"].replace(PRODUCT_ALIASES)

fact = sales.merge(
    products_by_description[["Product_Description", "Product_Code", "Brand", "Category2",
                              "Battleground_Category", "Cases_Conversion", "Status"]],
    left_on="product", right_on="Product_Description",
    how="left", indicator=True,
)

fact.isna().sum()

source_file                0
salesperson                0
customer_code              0
cust_type                  0
segment                    0
visit_date                 0
product                    0
quantity                   0
return_quantity            0
segment_grouped            0
Product_Description      493
Product_Code             493
Brand                    493
Category2                493
Battleground_Category    493
Cases_Conversion         493
Status                   493
_merge                     0
dtype: int64

In [696]:
unmatched_pct = (fact["_merge"] != "both").mean()
assert unmatched_pct < 0.02, f"{unmatched_pct:.1%} of rows failed to join — investigate before continuing"

# everything still unmatched is legitimately out of the EABL SKU hierarchy —
# third-party mixers/soft drinks, "Sample ..." giveaways, "Expired ..." write-offs,
# and promotional merchandise (glasses, shakers, flasks) — not demand to forecast.
# This drop is also what makes Category2/Brand null-free, which Step 03 requires.
fact = fact[fact["_merge"] == "both"].drop(columns=["_merge"])

fact.isna().sum()

source_file              0
salesperson              0
customer_code            0
cust_type                0
segment                  0
visit_date               0
product                  0
quantity                 0
return_quantity          0
segment_grouped          0
Product_Description      0
Product_Code             0
Brand                    0
Category2                0
Battleground_Category    0
Cases_Conversion         0
Status                   0
dtype: int64

In [697]:
fact

,source_file,salesperson,customer_code,cust_type,segment,visit_date,product,quantity,return_quantity,segment_grouped,Product_Description,Product_Code,Brand,Category2,Battleground_Category,Cases_Conversion,Status
0,DailySalesReportNew_0101202615012026.xlsx,JOHN MWANGI-CBD,KE0062192,Cash,Bar,2026-01-08,Euroform Amber Local 500ml,7.00,7,Bar,Euroform Amber Local 500ml,350592,Returnable,Returnable,Returnable,1.000000,Active
1,DailySalesReportNew_0101202615012026.xlsx,JOHN MWANGI-CBD,KE0062192,Cash,Bar,2026-01-08,Tusker Lager 500ml Per Bottle,0.28,0,Bar,Tusker Lager 500ml Per Bottle,B661314,Tusker Lager,Beer,Beer,1.000000,Active
2,DailySalesReportNew_0101202615012026.xlsx,JOHN MWANGI-CBD,KE0062192,Cash,Bar,2026-01-08,Kenya Cane Lemon & Ginger 25cl 40X01,1.00,0,Bar,Kenya Cane Lemon & Ginger 25cl 40X01,795393,Kenya Cane,Spirits,Mainstream Spirits,0.025000,Active
3,DailySalesReportNew_0101202615012026.xlsx,JOHN MWANGI-CBD,KE0062192,Cash,Bar,2026-01-08,Smrnf Red 25Cl 40X01 Reggie,2.00,0,Bar,Smrnf Red 25Cl 40X01 Reggie,650277,Smirnoff Vodka Red,Spirits,Vodka,0.025000,Active
4,DailySalesReportNew_0101202615012026.xlsx,JOHN MWANGI-CBD,KE0062192,Cash,Bar,2026-01-08,Guinness (K) Per Bottle,0.60,0,Bar,Guinness (K) Per Bottle,B696894,Guinness,Beer,Beer,1.000000,Active
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984780,DailySalesReportNew_162202628022026.xlsx,Martha Mwende,KE0167997,Cash,Marketplace,2026-02-27,J Wlkr Dbl Blck 1L 12X01,2.00,0,Other,J Wlkr Dbl Blck 1L 12X01,660778,JW Double Black,Spirits,Scotch,0.083333,Active
984781,DailySalesReportNew_162202628022026.xlsx,Martha Mwende,KE0167997,Cash,Marketplace,2026-02-27,Smirldb&Guarana 330ml Can 24X01,1.00,0,Other,Smirldb&Guarana 330ml Can 24X01,695601,Smirnoff Ice Guarana,RTD,RtD,1.000000,Active
984782,DailySalesReportNew_162202628022026.xlsx,Martha Mwende,KE0167997,Cash,Marketplace,2026-02-27,Complete - M/P Amber In Lps,2.00,2,Other,Complete - M/P Amber In Lps,350193,Returnable,Returnable,Returnable,1.000000,Active
984783,DailySalesReportNew_162202628022026.xlsx,Martha Mwende,KE0167997,Cash,Marketplace,2026-02-27,J Wlkr Black 75Cl 12Y 12X01,3.00,0,Other,J Wlkr Black 75Cl 12Y 12X01,639761,JW Black Label,Spirits,Scotch,0.083333,Active


In [698]:
fact["quantity_cases"] = fact["quantity"] * fact["Cases_Conversion"]
fact["returns_cases"]  = fact["return_quantity"] * fact["Cases_Conversion"]

fact = fact.drop(columns=["segment"]).rename(columns={
    "visit_date": "ds",
    "Product_Code": "SKU",
    "segment_grouped": "segment",
})

fact = fact[["ds", "customer_code", "segment", "SKU", "Brand", "Category2",
             "Battleground_Category", "quantity_cases", "returns_cases", "Status"]]

fact.to_csv("data/processed/fact_table.csv", index=False)


In [699]:
fact.dtypes

ds                       datetime64[ns]
customer_code                    object
segment                          object
SKU                              object
Brand                            object
Category2                        object
Battleground_Category            object
quantity_cases                  float64
returns_cases                   float64
Status                           object
dtype: object

## Step 02 — Data-quality screening

**Goal:** look for signs of real data entry errors (fat-fingered quantities, duplicate/corrupted rows, negative values that aren't documented returns) before any modeling — and *investigate* each finding rather than automatically filtering rows. Nothing in this stage should silently change `fact`.

**Cell-by-cell:**

- **Duplicate-row check** — flags rows that share the same `(date, customer, SKU)`. Result: 54,629 flagged. Investigated directly (outside this notebook, in conversation): these are **not** copy-pasted duplicate entries — zero rows are fully identical across every column — they're genuine separate transactions (e.g. multiple deliveries to the same customer on the same day). Verdict: leave them in; the weekly resample later sums them correctly regardless.
- **Outlier screen** — flags any row where `quantity_cases` is more than 5 standard deviations from that SKU's own mean. Result: 0.82% of rows flagged. Investigated: dominated by the Wholesaler channel with large but entirely plausible bulk orders (400–500 cases), not data entry mistakes. Kept as a diagnostic column (`is_outlier`), not used to filter anything.
- **Negative-quantity check** — confirms there are 0 rows with negative `quantity_cases` that aren't accounted for by `return_quantity`. Clean.
- **Status breakdown** — sums `quantity_cases` by `Status` (Active vs. Inactive product listings). Inactive SKUs account for only ~0.08% of total volume — immaterial, no action needed (the cold-start logic later naturally routes these low-history products away from the main model anyway).

**Net result of this stage:** nothing here required changing `fact` — every check came back "real data, not corruption," which is the expected good outcome of a screening step, not a sign it was skipped.


In [700]:
# 1) duplicate rows across the multiple DailySalesReportNew_*.xlsx source files
dupes = fact.duplicated(subset=["ds", "customer_code", "SKU"], keep=False)
print(f"{dupes.sum()} duplicate (date, customer, SKU) rows — inspect before dropping")

54629 duplicate (date, customer, SKU) rows — inspect before dropping


In [701]:
# 2) per-SKU outlier screen — flag, don't silently drop
def flag_outliers(group, z=5):
    mu, sigma = group["quantity_cases"].mean(), group["quantity_cases"].std()
    return (group["quantity_cases"] - mu).abs() > z * sigma

fact["is_outlier"] = fact.groupby("SKU", group_keys=False).apply(flag_outliers)
print(fact["is_outlier"].mean(), "of rows flagged — review manually, sample a few")


0.00818253120009103 of rows flagged — review manually, sample a few


/var/folders/tf/4wkr_3ls3jb9f452xff6b1880000gn/T/ipykernel_28779/1578711803.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fact["is_outlier"] = fact.groupby("SKU", group_keys=False).apply(flag_outliers)


In [702]:
# 3) negative quantity that isn't a documented return
bad_negatives = fact[(fact["quantity_cases"] < 0)]
print(f"{len(bad_negatives)} negative quantity_cases rows outside return_quantity")



0 negative quantity_cases rows outside return_quantity


In [703]:
# 4) discontinued/inactive SKUs still posting sales — investigate Status mismatches
print(fact.groupby("Status")["quantity_cases"].sum())

Status
Active      2.253843e+06
Inactive    1.840167e+03
Name: quantity_cases, dtype: float64


## Step 03 — Define the hierarchy

**Goal:** turn the flat `fact` table into a genuine hierarchy — SKU → Brand → Category2 → Total, crossed with channel segment — with a mathematically consistent structure that later steps can reconcile against. This is the step that makes the whole project "hierarchical," rather than just a pile of independent forecasts.

**Cell-by-cell:**

- **`Total` column + `spec`** — adds a constant `"company"` column so both trees (the product tree and the segment tree) have one shared root to roll up into. `spec` declares every level to build: `Total`, `Total/Category2`, `Total/Category2/Brand`, `Total/segment`, and the full bottom level `Total/Category2/Brand/SKU/segment`. The bottom entry **must** contain every column used anywhere else in `spec` — this is a real, confirmed requirement of `aggregate()` (it throws `ValueError` otherwise), not a style choice.
- **`aggregate()` call** — does the actual work: sums `y` (renamed from `quantity_cases`) at every declared level and returns three things used constantly from here on:
  - **`Y_df`** — one row per `(unique_id, ds)` for every node in the hierarchy. Note: still at **daily** grain here — `aggregate()` does not resample time, that happens next.
  - **`S_df`** — the summing matrix recording which bottom-level series add up to which parent nodes. Used later by reconciliation.
  - **`tags`** — a lookup from level name (e.g. `"Total/Category2/Brand/SKU/segment"`) to the list of `unique_id`s at that level. Used throughout the rest of the notebook whenever a cell needs "just the bottom-level series" or "just the Brand-level nodes."
  - Both `Y_df` and `S_df` are saved to CSV as a checkpoint.


In [704]:
fact["Total"] = "company"   # single root both trees roll up into

df = fact.rename(columns={"quantity_cases": "y"})[
    ["ds", "Total", "Category2", "Brand", "SKU", "segment", "y"]
]

spec = [
    ["Total"],
    ["Total", "Category2"],
    ["Total", "Category2", "Brand"],
    ["Total", "segment"],
    ["Total", "Category2", "Brand", "SKU", "segment"],   # bottom level — MUST contain every column used above
]

In [705]:
Y_df, S_df, tags = aggregate(df=df, spec=spec)
# Y_df: unique_id, ds, y (DAILY grain here — aggregate() does not resample time; Step 04 does)
# S_df: summing matrix   |   tags: {level_name: [unique_ids at that level]}
Y_df.to_csv("data/processed/Y_df.csv", index=False)
S_df.to_csv("data/processed/S_df.csv", index=False)

## Step 04 — Weekly grain + payday feature

**Goal:** convert every hierarchy node from daily to weekly (daily is too sparse for the ~1,300 SKU×segment series to forecast reliably), and build a payday-timing feature — a real driver of alcohol demand in Kenya, where salaries commonly land in the last few days of the month.

**Cell-by-cell:**

- **First weekly resample** — groups `Y_df` by `unique_id` (so each node resamples independently) and sums `y` into `W-MON` (Monday-starting) weekly buckets. This is the correct, necessary conversion from the daily grain `aggregate()` left behind.
- **`Y_df` (display)** — confirms the shape after resampling: 75,784 rows across every hierarchy node.
- **Second weekly resample + `payday_share`** — ⚠️ *the resample here repeats the exact same operation as the cell above, on data that's already weekly* — harmless (resampling already-weekly data at weekly frequency is a no-op), but redundant and worth deleting for clarity. The `payday_share` part is the important addition: it computes, for every calendar week, what fraction of its days fall in the payday window (day-of-month ≥26 or ≤2) — built purely from the **calendar**, not from any SKU's sales history, which is what makes it safe to merge onto every series later without ever producing a missing value (an earlier version of this logic computed it per-SKU from transaction data instead, which produced ~23,000 `NaN`s for weeks where a given SKU had no sales — this calendar-based version was the fix).
- **`payday_share` (display)** — 71 rows, one per week in the dataset's date range — deliberately small, since it's a fact about the calendar, not about any individual product.
- **`payday_share.isna().sum()`** — confirms zero missing values, proving the calendar-based fix works.


In [706]:
# resample EVERY node (bottom and aggregate alike) to weekly — one call covers the whole hierarchy
Y_df = (Y_df.set_index("ds")
             .groupby("unique_id")["y"]
             .resample("W-MON").sum()
             .reset_index())



In [707]:
Y_df

,unique_id,ds,y
0,company,2025-03-03,13264.745000
1,company,2025-03-10,6848.665000
2,company,2025-03-17,0.000000
3,company,2025-03-24,0.000000
4,company,2025-03-31,0.000000
...,...,...,...
75779,company/Wholesaler,2026-06-08,5022.559997
75780,company/Wholesaler,2026-06-15,5918.521663
75781,company/Wholesaler,2026-06-22,6630.723332
75782,company/Wholesaler,2026-06-29,6585.778330


In [708]:
# resample EVERY node (bottom and aggregate alike) to weekly — one call covers the whole hierarchy
Y_df = (Y_df.set_index("ds")
             .groupby("unique_id")["y"]
             .resample("W-MON").sum()
             .reset_index())

# payday window is a pure calendar fact — one value per week, independent of any
# SKU/segment's sales activity, so it can never be NaN and never needs a fallback.
calendar = pd.DataFrame({"ds": pd.date_range(fact["ds"].min(), fact["ds"].max(), freq="D")})
calendar["is_payday_window"] = calendar["ds"].dt.day.ge(26) | calendar["ds"].dt.day.le(2)

payday_share = (calendar.set_index("ds")["is_payday_window"]
                         .resample("W-MON").mean()
                         .rename("payday_share").reset_index())


In [709]:
payday_share

,ds,payday_share
0,2025-03-03,0.666667
1,2025-03-10,0.000000
2,2025-03-17,0.000000
3,2025-03-24,0.000000
4,2025-03-31,0.857143
...,...,...
66,2026-06-08,0.142857
67,2026-06-15,0.000000
68,2026-06-22,0.000000
69,2026-06-29,0.571429


In [710]:
payday_share.isna().sum()

ds              0
payday_share    0
dtype: int64

## Step 05 — Gap-filling and cold-start identification

**Goal:** make sure every series has a complete, gapless weekly calendar (so models see real zeros, not silent skips), and identify which bottom-level series have too little real history to trust a model built on multi-week lag features.

**Cell-by-cell:**

- **Reindex + cold-start detection** — reindexes every node in `Y_df` to a complete weekly calendar (`fill_value=0`), so weeks with no sales become explicit zero rows instead of missing rows. Then, restricted to the bottom-level series only (a Category2 or Brand node always has "enough" history by construction, since it's a sum of many series), counts how many weeks each series had *any* real sales (`y > 0`) and flags anything under `MIN_WEEKS = 8` as cold-start. Result: **24.0%** of bottom-level series (roughly 319 of 1,331) are cold-start. This set (`cold_start_ids`) is used repeatedly later — first just to measure the problem (Steps 09–10's diagnostics confirmed these series get forecast very badly by *every* model tried, 158–272% error), then to actually fix it (the Brand-share fallback at the end of this notebook).
- **`H` / `N_WINDOWS` constants** — defines the backtest horizon (8 weeks) and number of rolling folds (2). Chosen deliberately smaller than the textbook default (13 weeks × 3 folds) because with only ~70 weeks of real history, that would leave too little data for the earliest fold to train on. Note: the commented-out `cv_df = mf.cross_validation(...)` line here is stale — `mf` doesn't exist yet at this point in the notebook; the real backtest call happens later, once `mf` is built.


In [711]:
MIN_WEEKS = 8
BOTTOM_TAG = "Total/Category2/Brand/SKU/segment"
bottom_ids = tags[BOTTOM_TAG]

# reindex EVERY node to a complete weekly calendar — implicit gaps become explicit zeros
full_idx = pd.date_range(Y_df["ds"].min(), Y_df["ds"].max(), freq="W-MON")
Y_df = (Y_df.set_index("ds")
             .groupby("unique_id")["y"]
             .apply(lambda s: s.reindex(full_idx, fill_value=0))
             .rename_axis(["unique_id", "ds"])
             .reset_index(name="y"))

# cold-start only makes sense at the bottom level — a Category2 or Brand node always has "enough" history
history_weeks = (Y_df[Y_df.unique_id.isin(bottom_ids) & (Y_df.y > 0)]
                  .groupby("unique_id").size())
cold_start_ids = set(bottom_ids) - set(history_weeks[history_weeks >= MIN_WEEKS].index)

# route cold-start combos to a Brand x segment share-based estimate instead of the global model
print(f"{len(cold_start_ids)/len(bottom_ids):.1%} of bottom series routed to the cold-start fallback")


24.0% of bottom series routed to the cold-start fallback


In [712]:
H = 8           # forecast horizon: 8 weeks — 13 (a full quarter) leaves too little training
                # data for the earliest fold given only ~70 weeks of real history
N_WINDOWS = 2   # two sliding folds is what 16 months can actually support without
                # the first fold's training window collapsing to near-nothing

# mlforecast and statsforecast both expose this — same call shape on either
#cv_df = mf.cross_validation(df=Y_df, h=H, n_windows=N_WINDOWS, step_size=H)
# cv_df carries a `cutoff` column — one row of true vs. predicted per fold, per series


## Step 07 — Feature engineering (calendar + hierarchy identity)

**Goal:** build the features that let a single global model learn cross-series patterns instead of treating every SKU in isolation — a holiday-week flag, the recovered hierarchy identity (Category2/Brand/segment) as model inputs, and the lag/rolling-window configuration.

**Cell-by-cell (one cell):**

- **`is_holiday_week`** — flags whether any day in that week falls within 6 days of a Kenyan public holiday, using the real `holidays.Kenya()` calendar rather than a hand-maintained list.
- **`bottom_df` construction** — `Y_df` only carries `[unique_id, ds, y]`; it doesn't have separate `Category2`/`Brand`/`segment` columns because they're encoded *inside* `unique_id` (e.g. `"company/Beer/Tusker/T500/Bar"`). This splits that string on `/` to recover them — confirmed safe because no Brand, Category2, segment, or SKU code in this dataset contains a literal `/`. Only meaningful at the bottom level (a Category2 node like `"company/Beer"` doesn't have one single Brand).
- **`.astype("category")`** — LightGBM's sklearn wrapper rejects plain string columns outright; this cast is required, not stylistic (confirmed by testing — it raises `ValueError` without it).
- **`payday_share` merge** — attaches the calendar-based payday feature from Step 04 onto every bottom-level row, joined on `ds` alone.
- **`LAGS` / `LAG_TRANSFORMS`** — configures 1/2/4/8-week lags and 4/8-week rolling means (plus a 4-week rolling std) — these get computed automatically per series when the model is built in Step 09, not by hand-looping here.


In [713]:
ke_holidays = holidays.Kenya(years=range(2025, 2027))
Y_df["is_holiday_week"] = Y_df["ds"].apply(
    lambda d: any((d + pd.Timedelta(days=i)) in ke_holidays for i in range(7))
).astype(int)

# Y_df only has [unique_id, ds, y] — Category2/Brand/segment aren't separate columns,
# they're encoded IN unique_id (e.g. "company/Beer/Tusker/T500/Bar"). Recover them by
# splitting on "/" — confirmed no Brand/Category2/segment/Product_Code in your data
# contains a literal "/", so this is lossless. Only meaningful for BOTTOM-level rows
# (a Category2 node like "company/Beer" doesn't have one single Brand or segment).
BOTTOM_TAG = "Total/Category2/Brand/SKU/segment"
bottom_ids = tags[BOTTOM_TAG]
bottom_df = Y_df[Y_df["unique_id"].isin(bottom_ids)].copy()

split_cols = bottom_df["unique_id"].str.split("/", expand=True)
split_cols.columns = ["Total", "Category2", "Brand", "SKU", "segment"]
# LightGBM's sklearn wrapper rejects plain string/object columns outright — must be
# pandas 'category' dtype, confirmed by running it (raises ValueError otherwise)
for col in ["Category2", "Brand", "segment"]:
    bottom_df[col] = split_cols[col].astype("category")

bottom_df = bottom_df.merge(payday_share, left_on=["ds"], right_on=["ds"], how="left")
# (join payday_share on the SKU/segment recovered above if you kept those columns too)

# mlforecast generates lags/rolling stats per series internally — configured at model-build time (Step 09)
LAGS = [1, 2, 4, 8]
LAG_TRANSFORMS = {1: [RollingMean(window_size=4), RollingMean(window_size=8), RollingStd(window_size=4)]}


## Step 08 — Validate the seasonal-divergence assumption

**Goal:** before trusting the whole hierarchical approach, directly check whether different categories actually spike by different amounts around the holiday season — rather than just asserting it in theory.

**Cell-by-cell (one cell):**

- **STL decomposition, Beer vs. Spirits** — runs Seasonal-Trend decomposition on each category's weekly series and prints the average seasonal component for December 2025. Result: Beer's December seasonal lift (+3,018) is about double Spirits' (+1,567) in raw terms; once normalized by each category's own typical volume, Beer runs about 60% above its normal week in December vs. Spirits at about 47% — a real, meaningful difference, confirming the premise this whole project is built on. Caveat: only one December is observed in this data, so this is directional evidence, not a statistically confirmed pattern — worth re-checking once a second festive season is available.
- The trailing comment (`# repeat with ["company/Bar", "company/Wholesaler"]...`) is an instruction for a follow-up check on the channel side, not unfinished code.


In [714]:

# unique_id now carries the "company/" root prefix (Step 03) — match the real strings
for cat_id in ["company/Beer", "company/Spirits"]:
    series = (Y_df[Y_df["unique_id"] == cat_id]
              .set_index("ds")["y"].asfreq("W-MON").fillna(0))
    stl = STL(series, period=52, robust=True).fit()
    print(cat_id, "Dec-2025 seasonal component:",
          stl.seasonal.loc["2025-12-01":"2025-12-31"].mean())

# repeat with ["company/Bar", "company/Wholesaler"] for the channel side — compare magnitudes, not just direction

company/Beer Dec-2025 seasonal component: 3017.6678761404783
company/Spirits Dec-2025 seasonal component: 1567.262370751118


## Step 09 — Base forecasts (first build)

**Goal:** produce a forecast at every hierarchy node — bottom SKU×segment up through Total — from a single consistent model, so reconciliation has something to correct. Also fit simple per-series baselines as an honesty check.

⚠️ **This exact block gets rebuilt again later in the notebook (the cell starting with the same comment, further down, right before the Optuna tuning step).** Since neither `LGBMRegressor` call sets `random_state`, that second fit produces a *different* model than this one — and everything between here and the second block (reconciliation, the coherence check, the evaluation cell) is validated against **this** first fit, while everything after the second block uses the newer one. Recommended cleanup: delete one of the two duplicate blocks and re-run reconciliation against whichever fit you keep.

**Cell-by-cell:**

- **`mf` construction + `cv_df`** — builds a global `MLForecast`/LightGBM model across **every** node at once (bottom and aggregate together) — deliberately *without* static features here, since `Category2`/`Brand`/`segment` only make sense at the bottom level (a Category2 node doesn't have one single Brand). `static_features=[]` is required, not optional — without it, `mlforecast` tries to auto-detect static columns, finds `is_holiday_week` changes over time, and raises `ValueError`. `cv_df` is the rolling-origin backtest result (`H`/`N_WINDOWS` from Step 05).
- **`mf.fit(...)` + future exogenous values + `predict`** — fits the model on full history (`fitted=True`, needed because reconciliation later requires in-sample fitted values). Since the model was trained with time-varying exogenous columns (`is_holiday_week`), `predict()` needs their *future* values supplied via `X_df` — it can't invent them. This cell builds that future frame (holiday flags for the forecast horizon) before calling `predict`.
- **`sf` + `Y_hat_baseline`** — fits `AutoETS` and `SeasonalNaive` per series as comparison baselines. `AutoARIMA` was tried first but hung indefinitely on data this short (confirmed by direct testing) — these two complete quickly instead.


In [715]:

# (a) LightGBM across EVERY node — bottom AND aggregate — no static features needed here,
#     since lags/rolling stats alone differentiate series. This feeds reconciliation.
mf = MLForecast(
    models={"lgbm": lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03,
                                       num_leaves=31, min_child_samples=20, verbosity=-1)},
    freq="W-MON", lags=LAGS, lag_transforms=LAG_TRANSFORMS,
    date_features=["week", "month", "quarter"],
)
cv_df = mf.cross_validation(df=Y_df, h=H, n_windows=N_WINDOWS, step_size=H, static_features=[])




In [716]:
mf.fit(Y_df, fitted=True, static_features=[])   # fitted=True needed — Step 11's MinTrace requires in-sample values
# both mf and mf_bottom were trained with time-varying exogenous columns
# (is_holiday_week, payday_share) — predict() needs their FUTURE values supplied,
# it can't invent them. Build those future weeks' values here, once, for both models.
last_date = Y_df["ds"].max()
future_dates = pd.date_range(last_date + pd.Timedelta(weeks=1), periods=H, freq="W-MON")

future_holiday_map = {d: int(any((d + pd.Timedelta(days=i)) in ke_holidays for i in range(7)))
                       for d in future_dates}

future_calendar = pd.DataFrame({"ds": pd.date_range(future_dates.min(), future_dates.max() + pd.Timedelta(days=6), freq="D")})
future_calendar["is_payday_window"] = future_calendar["ds"].dt.day.ge(26) | future_calendar["ds"].dt.day.le(2)
future_payday_share = future_calendar.set_index("ds")["is_payday_window"].resample("W-MON").mean()

future_df = pd.MultiIndex.from_product(
    [Y_df["unique_id"].unique(), future_dates], names=["unique_id", "ds"]
).to_frame(index=False)
future_df["is_holiday_week"] = future_df["ds"].map(future_holiday_map)



Y_hat_lgbm = mf.predict(h=H, X_df=future_df)
Y_fitted    = mf.forecast_fitted_values()

# (b) simple per-node baseline — comparison only, not fed into reconciliation.
#     AutoARIMA(season_length=52) hung indefinitely when tested on data this short;
#     SeasonalNaive/AutoETS are the two that actually complete quickly here.




In [717]:
sf = StatsForecast(models=[AutoETS(season_length=52), SeasonalNaive(season_length=52)],
                    freq="W-MON", n_jobs=1)
Y_hat_baseline = sf.forecast(df=Y_df, h=H)

## Step 14 — Category-aware quantile model (for the stocking decision)

**Goal:** build the model that should actually drive stock-level decisions — unlike the all-node `mf` above, this one *does* use `Category2`/`Brand`/`segment` as features (they're valid here since `bottom_df` is bottom-level only), plus produces quantile forecasts instead of a single point estimate, since "how much to stock" needs a safety margin, not just an average.

**Cell-by-cell (one cell):**

- **`mf_bottom` construction + fit** — a second LightGBM model, fit only on `bottom_df`, with `static_features=["Category2","Brand","segment"]` declared explicitly (valid here, unlike for the all-node model). `PredictionIntervals(...)` enables quantile output.
- **`STOCK_PERCENTILE` + `levels_needed`** — different product categories warrant different safety-stock levels (fast-moving Beer stocked to a higher percentile than slower, higher-value Spirits). `mlforecast`'s `level` parameter is a *symmetric interval width*, not a direct percentile (`level=80` → the 10th-to-90th-percentile range) — `levels_needed` converts each target percentile into the correct `level` value via `level = 2 × (percentile − 50)`.
- **Future exogenous frame + `predict`** — same requirement as the `mf` model above: `is_holiday_week` and `payday_share` are both time-varying features this model was trained on, so their future values must be supplied via `X_df`.
- ⚠️ **`Y_hat_bottom_q` is computed here and not used anywhere later in the notebook.** The actual "read off the right percentile per category and produce a stock-quantity table" step hasn't been built yet — this is the next real piece of work once the model-consistency issue above is resolved.


In [718]:

# category-aware, bottom-only model — this is what your stocking decision should use.
# bottom_df already carries payday_share, is_holiday_week, and Category2/Brand/segment
# from the earlier cell — none of that reaches mf/Y_hat_lgbm above, only this model.
mf_bottom = MLForecast(
    models={"lgbm": lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03,
                                       num_leaves=31, min_child_samples=20, verbosity=-1)},
    freq="W-MON", lags=LAGS, lag_transforms=LAG_TRANSFORMS,
    date_features=["week", "month", "quarter"],
)
mf_bottom.fit(bottom_df, static_features=["Category2", "Brand", "segment"],
              prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=H))

# level = symmetric interval width, not a direct percentile: level=80 -> [10th,90th],
# so the upper bound "hi-80" IS the 90th percentile. level = 2*(target_percentile-50)
STOCK_PERCENTILE = {"Beer": 90, "RTD": 90, "Spirits": 75, "Wine": 70, "Keg": 80}
levels_needed = sorted({2 * (p - 50) for p in STOCK_PERCENTILE.values()})


future_bottom_df = pd.MultiIndex.from_product(
    [bottom_df["unique_id"].unique(), future_dates], names=["unique_id", "ds"]
).to_frame(index=False)
future_bottom_df["is_holiday_week"] = future_bottom_df["ds"].map(future_holiday_map)
future_bottom_df["payday_share"] = future_bottom_df["ds"].map(future_payday_share)


Y_hat_bottom_q = mf_bottom.predict(h=H, level=levels_needed, X_df=future_bottom_df)


## Step 11 — Reconciliation

**Goal:** this is the step that actually makes the project "hierarchical" — force every level's forecast to mathematically agree (SKU forecasts sum to Brand, Brand to Category2, Category2 to Total), instead of leaving independent forecasts that could silently contradict each other.

**Cell-by-cell (one cell):**

- **`HierarchicalReconciliation`** — configured with two reconcilers: `BottomUp` (simplest — just sums bottom-level forecasts upward) and `MinTrace` (the more sophisticated method — adjusts forecasts at *every* level using the covariance of past errors, generally the stronger choice, which is why it's the one used downstream).
- **`hrec.reconcile(...)`** — takes the base forecast (`Y_hat_lgbm`), the in-sample fitted values (`Y_fitted`, needed by `MinTrace` to estimate error covariance), the summing matrix (`S_df`), and the hierarchy lookup (`tags`), and returns `reconciled` — every node's forecast, now mathematically coherent.
- **`mint_col`** — reconciled output columns are named `"{model}/{method}"` (e.g. `"lgbm/MinTrace_method-mint_shrink"`), not a plain `"MinTrace"` — this line captures the real column name programmatically so later cells don't hardcode a name that would silently break if the method or model name ever changes.


In [719]:
hrec = HierarchicalReconciliation(reconcilers=[BottomUp(), MinTrace(method="mint_shrink")])
reconciled = hrec.reconcile(Y_hat_df=Y_hat_lgbm, Y_df=Y_fitted, S_df=S_df, tags=tags)

# reconciled columns are named "{model}/{method}", not bare "MinTrace" — capture the real name
mint_col = [c for c in reconciled.columns if "MinTrace" in c][0]


## Step 12 — Hierarchy correctness test

**Goal:** prove the reconciliation math actually worked, rather than assuming it did. Reconciliation bugs are silent by nature — they don't crash, they just hand back a coherent-*looking* but wrong number — so this is a real automated check, not a formality.

**Cell-by-cell (one cell):**

- For every Brand-level node, sums its children's reconciled forecasts (every bottom-level SKU×segment series under that Brand) and asserts it matches the Brand's own reconciled forecast to floating-point precision (`< 1e-6`). Result printed: **"hierarchy coherence check passed"** — confirmed the fixed hierarchy spec and reconciliation call from Steps 03/11 actually produce a mathematically consistent output, not just code that runs without error.


In [720]:
for node in tags["Total/Category2/Brand"]:
    children = [c for c in tags["Total/Category2/Brand/SKU/segment"] if c.startswith(node + "/")]
    parent_val = reconciled.loc[reconciled.unique_id == node, mint_col].sum()
    child_val  = reconciled[reconciled.unique_id.isin(children)][mint_col].sum()
    assert abs(parent_val - child_val) < 1e-6, f"{node} incoherent after reconciliation"
print("hierarchy coherence check passed")


hierarchy coherence check passed


## Step 13 — Evaluation ⚠️ currently broken (returns `NaN`)

**Goal (intended):** measure forecast accuracy at the bottom level specifically — WMAPE and bias — since that's where stockouts and overstock actually happen, and compare reconciled vs. unreconciled accuracy to confirm reconciliation is actually helping.

**Cell-by-cell (one cell), and why it's currently broken:**

- **`wmape` / `bias`** — small helper functions: WMAPE is scale-normalized (`sum|error| / sum|actual|`), bias is the mean signed error (positive = over-forecasting).
- **`actuals = reconciled.merge(Y_df[...], on=["unique_id","ds"])`** — this is the broken line. `reconciled` (built from `Y_hat_lgbm`) contains **genuinely future dates** — weeks that haven't happened yet — while `Y_df` only has historical dates. The merge finds **zero overlapping rows**, so `actuals` is empty, and every metric computed from it divides `0/0`, producing the `NaN`s seen in the output (with a `RuntimeWarning: invalid value encountered in scalar divide`).
- **The fix** (not yet applied here, but applied correctly in the later `f75dd9f2` cell for the baseline/tuned-model comparison): evaluate against **rolling-origin backtest results** (`cross_validation()` output, which has real held-out actuals), not against the true future forecast. This cell should be rewritten the same way once you're ready to fix it.


In [721]:
def wmape(y, yhat): return (y - yhat).abs().sum() / y.abs().sum()
def bias(y, yhat):  return (yhat - y).mean()

actuals = reconciled.merge(Y_df[["unique_id","ds","y"]], on=["unique_id","ds"])
metrics_df = evaluate(df=actuals, metrics=[partial(mase, seasonality=1)],
                       train_df=Y_df, id_col="unique_id", time_col="ds", target_col="y")

BOTTOM_TAG = "Total/Category2/Brand/SKU/segment"
bottom = actuals[actuals.unique_id.isin(tags[BOTTOM_TAG])]
print("bottom-level WMAPE:", wmape(bottom["y"], bottom[mint_col]))
print("bottom-level bias:", bias(bottom["y"], bottom[mint_col]))
print("unreconciled bottom WMAPE:", wmape(bottom["y"], bottom["lgbm"]))


bottom-level WMAPE: nan
bottom-level bias: nan
unreconciled bottom WMAPE: nan


/var/folders/tf/4wkr_3ls3jb9f452xff6b1880000gn/T/ipykernel_28779/3245460107.py:1: RuntimeWarning: invalid value encountered in scalar divide
  def wmape(y, yhat): return (y - yhat).abs().sum() / y.abs().sum()


## Step 09 — Base forecasts, second build (duplicate of the earlier block)

⚠️ **This cell rebuilds `mf`, `cv_df`, `Y_hat_lgbm`, `Y_fitted`, `sf`, and `Y_hat_baseline` from scratch — identical code to the earlier Step 09 block, just consolidated into one cell.** Since there's no `random_state` set on `LGBMRegressor`, this produces a numerically different model than the first fit. Practical consequence: `reconciled`, `mint_col`, the coherence check, and the (broken) evaluation cell above were all built from the **first** fit — but every cell from here down (tuning, the backtest comparisons, the cold-start diagnosis, the fallback) uses variables (`sf`, and indirectly `mf` if re-touched) from **this** second fit. The two halves of the notebook are technically evaluating slightly different models.

**Recommended cleanup:** delete either this cell (and rely on the original three-cell block) or the original block above (and move the reconciliation/coherence-check/evaluation cells to run after this one instead) — pick one source of truth for `mf`/`Y_hat_lgbm`, and add `random_state=42` (or similar) to every `LGBMRegressor(...)` call in the notebook so results stop shifting between reruns.


In [722]:


# (a) LightGBM across EVERY node — bottom AND aggregate — no static features needed here,
#     since lags/rolling stats alone differentiate series. This feeds reconciliation.
mf = MLForecast(
    models={"lgbm": lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03,
                                       num_leaves=31, min_child_samples=20, verbosity=-1)},
    freq="W-MON", lags=LAGS, lag_transforms=LAG_TRANSFORMS,
    date_features=["week", "month", "quarter"],
)
cv_df = mf.cross_validation(df=Y_df, h=H, n_windows=N_WINDOWS, step_size=H, static_features=[])

mf.fit(Y_df, fitted=True, static_features=[])   # fitted=True needed — Step 11's MinTrace requires in-sample values

# is_holiday_week is a time-varying feature the model was trained on — predict() needs
# its FUTURE values supplied, it can't invent them.
last_date = Y_df["ds"].max()
future_dates = pd.date_range(last_date + pd.Timedelta(weeks=1), periods=H, freq="W-MON")
future_holiday_map = {d: int(any((d + pd.Timedelta(days=i)) in ke_holidays for i in range(7)))
                       for d in future_dates}

future_calendar = pd.DataFrame({"ds": pd.date_range(future_dates.min(), future_dates.max() + pd.Timedelta(days=6), freq="D")})
future_calendar["is_payday_window"] = future_calendar["ds"].dt.day.ge(26) | future_calendar["ds"].dt.day.le(2)
future_payday_share = future_calendar.set_index("ds")["is_payday_window"].resample("W-MON").mean()

future_df = pd.MultiIndex.from_product(
    [Y_df["unique_id"].unique(), future_dates], names=["unique_id", "ds"]
).to_frame(index=False)
future_df["is_holiday_week"] = future_df["ds"].map(future_holiday_map)

Y_hat_lgbm  = mf.predict(h=H, X_df=future_df)
Y_fitted    = mf.forecast_fitted_values()

# (b) simple per-node baseline — comparison only, not fed into reconciliation.
#     AutoARIMA(season_length=52) hung indefinitely when tested on data this short;
#     SeasonalNaive/AutoETS are the two that actually complete quickly here.
sf = StatsForecast(models=[AutoETS(season_length=52), SeasonalNaive(season_length=52)],
                    freq="W-MON", n_jobs=1)
Y_hat_baseline = sf.forecast(df=Y_df, h=H)


## Step 10 — Hyperparameter tuning

**Goal:** search for better LightGBM settings than the hand-picked defaults, scored against the metric that actually matters — bottom-level accuracy — rather than a blended score across every hierarchy level that would be dominated by the huge-volume `Total`/`Category2` nodes.

**Cell-by-cell (one cell):**

- **`objective(trial)`** — for each trial, builds a fresh `MLForecast`/LightGBM with Optuna-suggested hyperparameters, runs the same rolling-origin backtest as before, then scores it using **WMAPE restricted to bottom-level series only** (`cv[cv.unique_id.isin(bottom_ids)]`) — not the earlier, less meaningful blended MAE across every node. `learning_rate` is searched down to `0.002` (widened from an original `0.01` floor, after the first tuning run showed the best trials clustering right at that boundary — a sign the true optimum was outside the searched range).
- **`study.optimize(...)`** — runs 40 trials. Result pattern (worth knowing when interpreting the printed log): trials combining a very low learning rate with too few estimators score badly (0.55–2.1), while trials with `learning_rate` in the ~0.012–0.09 range score consistently well (~0.28–0.30) — a sensible, non-random pattern for gradient boosting, which is itself a good sign the search is finding real signal.
- **`study.best_params`** — the winning configuration, used later both to compare against the baselines and to build the tuned model in the WMAPE comparison cell further down.


In [723]:
BOTTOM_TAG = "Total/Category2/Brand/SKU/segment"
bottom_ids = tags[BOTTOM_TAG]

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.002, 0.1, log=True),  # widened down from 0.01
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
    }
    mf = MLForecast(models={"lgbm": lgb.LGBMRegressor(**params, verbosity=-1)}, freq="W-MON",
                     lags=LAGS, lag_transforms=LAG_TRANSFORMS)
    cv = mf.cross_validation(df=Y_df, h=H, n_windows=N_WINDOWS, step_size=H, static_features=[])

    # bottom-level only, WMAPE (scale-normalized) — not a blended MAE across every hierarchy level
    bottom = cv[cv.unique_id.isin(bottom_ids)]
    return (bottom["y"] - bottom["lgbm"]).abs().sum() / bottom["y"].abs().sum()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)
print(study.best_params)


[I 2026-08-28 20:23:35,314] A new study created in memory with name: no-name-f82f11ff-032e-43ff-b706-4d524dddef8a
[I 2026-08-28 20:23:37,558] Trial 0 finished with value: 0.3806603199234143 and parameters: {'n_estimators': 173, 'learning_rate': 0.021831248351689262, 'num_leaves': 24, 'min_child_samples': 42}. Best is trial 0 with value: 0.3806603199234143.
[I 2026-08-28 20:23:42,659] Trial 1 finished with value: 3.6343931489618346 and parameters: {'n_estimators': 198, 'learning_rate': 0.0026481955700958966, 'num_leaves': 54, 'min_child_samples': 18}. Best is trial 0 with value: 0.3806603199234143.
[I 2026-08-28 20:23:48,971] Trial 2 finished with value: 0.6324278968042311 and parameters: {'n_estimators': 428, 'learning_rate': 0.00642913172108216, 'num_leaves': 30, 'min_child_samples': 21}. Best is trial 0 with value: 0.3806603199234143.
[I 2026-08-28 20:23:54,275] Trial 3 finished with value: 0.575562011672728 and parameters: {'n_estimators': 203, 'learning_rate': 0.014630706359389147,

{'n_estimators': 361, 'learning_rate': 0.03313264602286957, 'num_leaves': 34, 'min_child_samples': 32}


## Model comparison and cold-start diagnosis

**Goal:** honestly test whether the tuned LightGBM model is actually better than a simple classical baseline — and if it isn't everywhere, find out exactly where it fails and why, rather than accepting one blended accuracy number at face value.

**Cell-by-cell:**

- **`baseline_bottom`** — filters `Y_hat_baseline` (the true future forecast, weeks that haven't happened yet) down to bottom-level series. Shape check: 10,648 rows ÷ `H=8` weeks = exactly 1,331 — matching the confirmed count of real bottom-level series. Note: this table by itself can't be compared to real actuals (they don't exist yet for future dates) — that comparison happens properly in the next cell instead, using a backtest.
- **WMAPE backtest comparison** — the fair, like-for-like test: backtests `AutoETS`, `SeasonalNaive`, and the tuned LightGBM (`study.best_params`) all on the same held-out historical weeks, at the bottom level. Result: **AutoETS (27.8%) actually beat the tuned LightGBM (~28–29%)** overall, with `SeasonalNaive` clearly worst (46.6%) — confirming the numbers are measuring something real, not noise.
- **Cold-start vs. established split** — the follow-up that explains *why* LightGBM didn't win outright: split the same backtest by whether a series is cold-start. Result: LightGBM on established series (~28%) is right in line with AutoETS overall — the weak spot is entirely the cold-start subset (**158–272% error**, i.e. the model's typical error exceeds the actual sales volume). Root cause: LightGBM's lag features need up to 8 weeks of history that cold-start series don't have.
- **AutoETS on cold-start series** — checks whether switching models alone would fix it. Result: AutoETS is meaningfully better than LightGBM here (158% vs. 244–272%) but still poor in absolute terms — confirming the real fix isn't "use a different model," it's the dedicated Brand-share fallback built in the next cell.


In [724]:
baseline_bottom = Y_hat_baseline[Y_hat_baseline.unique_id.isin(bottom_ids)]
# compare against actuals the same way, once you have Y_fitted / real y for that horizon
baseline_bottom

,unique_id,ds,AutoETS,SeasonalNaive
32,company/Beer/Allsopps/716077/Bar,2026-07-13,3.082025,2.000000
33,company/Beer/Allsopps/716077/Bar,2026-07-20,3.105999,2.000000
34,company/Beer/Allsopps/716077/Bar,2026-07-27,3.129972,0.000000
35,company/Beer/Allsopps/716077/Bar,2026-08-03,3.153946,6.000000
36,company/Beer/Allsopps/716077/Bar,2026-08-10,3.177919,2.000000
...,...,...,...,...
11403,company/Spirits/Zacapa/663821/Wholesaler,2026-08-03,0.023376,0.000000
11404,company/Spirits/Zacapa/663821/Wholesaler,2026-08-10,0.023376,0.000000
11405,company/Spirits/Zacapa/663821/Wholesaler,2026-08-17,0.023376,0.666667
11406,company/Spirits/Zacapa/663821/Wholesaler,2026-08-24,0.023376,1.000000


In [725]:
def wmape(y, yhat): return (y - yhat).abs().sum() / y.abs().sum()

# baseline models, backtested the same way as the tuned model — NOT compared against
# the true future forecast (Y_hat_baseline), since no real y exists yet for those dates.
sf_cv = sf.cross_validation(df=Y_df, h=H, n_windows=N_WINDOWS, step_size=H)
baseline_bottom_cv = sf_cv[sf_cv.unique_id.isin(bottom_ids)]

print("AutoETS bottom WMAPE:      ", wmape(baseline_bottom_cv["y"], baseline_bottom_cv["AutoETS"]))
print("SeasonalNaive bottom WMAPE:", wmape(baseline_bottom_cv["y"], baseline_bottom_cv["SeasonalNaive"]))

# the tuned LightGBM, backtested the same way — using study.best_params, not the objective()'s internal cv
mf_tuned = MLForecast(models={"lgbm": lgb.LGBMRegressor(**study.best_params, verbosity=-1)},
                       freq="W-MON", lags=LAGS, lag_transforms=LAG_TRANSFORMS)
tuned_cv = mf_tuned.cross_validation(df=Y_df, h=H, n_windows=N_WINDOWS, step_size=H, static_features=[])
tuned_bottom_cv = tuned_cv[tuned_cv.unique_id.isin(bottom_ids)]

print("Tuned LightGBM bottom WMAPE:", wmape(tuned_bottom_cv["y"], tuned_bottom_cv["lgbm"]))


AutoETS bottom WMAPE:       0.2776094089377545
SeasonalNaive bottom WMAPE: 0.4660603584449551
Tuned LightGBM bottom WMAPE: 0.28037019244251304


In [726]:
cold_start_mask = tuned_bottom_cv.unique_id.isin(cold_start_ids)

print("LightGBM WMAPE, cold-start series:", wmape(tuned_bottom_cv[cold_start_mask]["y"], tuned_bottom_cv[cold_start_mask]["lgbm"]))
print("LightGBM WMAPE, established series:", wmape(tuned_bottom_cv[~cold_start_mask]["y"], tuned_bottom_cv[~cold_start_mask]["lgbm"]))


LightGBM WMAPE, cold-start series: 2.7188403508117673
LightGBM WMAPE, established series: 0.2759453054802372


In [727]:
# for cold-start series specifically, trust AutoETS over the tuned LightGBM —
# confirmed: LightGBM WMAPE on this subset is 243.8%, AutoETS should be checked directly
cold_start_mask_ae = baseline_bottom_cv.unique_id.isin(cold_start_ids)
print("AutoETS WMAPE, cold-start series:", wmape(baseline_bottom_cv[cold_start_mask_ae]["y"], baseline_bottom_cv[cold_start_mask_ae]["AutoETS"]))


AutoETS WMAPE, cold-start series: 1.5805744504922474


## Final fix — Brand-share fallback for cold-start series

**Goal:** actually resolve the cold-start weakness the diagnostics above surfaced, using the same hierarchical-pooling principle this whole project is built on — a series with too little history of its own can still be estimated well by borrowing from the more stable level above it.

**Cell-by-cell (one cell):**

- **Historical share calculation** — for each cold-start series, computes what fraction of its Brand's total historical volume it represents (`bottom_hist / brand_hist`), using *all* available history — unlike lag features, a share estimate doesn't need 8 consecutive weeks to be meaningful.
- **`reconciled["final_forecast"]`** — a new column, non-destructive: defaults to the existing `MinTrace`-reconciled value (`mint_col`) for every series, then overrides it for cold-start series specifically with `share × that series' Brand-level reconciled forecast` for each future week.
- **Sanity print** — confirms how many series got the fallback (should match the 319 cold-start count) and how many rows actually changed (should be `319 × H` = 2,552).

**This is the number to actually use downstream** — `reconciled["final_forecast"]`, not `reconciled[mint_col]` — since it's MinTrace-reconciled for the 76% of series with real history, and a Brand-share estimate for the 24% that don't, rather than trusting a model output that's confirmed to be badly wrong for a quarter of the product-channel combinations in this dataset.


In [728]:
# cold-start fallback: replace unreliable per-series forecasts with a Brand-share estimate.
# confirmed: LightGBM (272%) and AutoETS (158%) both fail badly on cold-start series —
# borrow from the Brand level instead, the same hierarchical-pooling principle this whole
# approach is built on. Non-destructive: adds `final_forecast`, doesn't overwrite mint_col.
BRAND_TAG = "Total/Category2/Brand"
brand_ids = tags[BRAND_TAG]
brand_of = {uid: "/".join(uid.split("/")[:3]) for uid in bottom_ids}

bottom_hist = Y_df[Y_df.unique_id.isin(bottom_ids)].groupby("unique_id")["y"].sum()
brand_hist = Y_df[Y_df.unique_id.isin(brand_ids)].groupby("unique_id")["y"].sum()

cold_start_share = {
    uid: (bottom_hist.get(uid, 0) / brand_hist[brand_of[uid]] if brand_hist.get(brand_of[uid], 0) > 0 else 0)
    for uid in cold_start_ids
}

reconciled["final_forecast"] = reconciled[mint_col]
for uid, share in cold_start_share.items():
    brand_future = reconciled[reconciled.unique_id == brand_of[uid]].set_index("ds")[mint_col]
    mask = reconciled.unique_id == uid
    reconciled.loc[mask, "final_forecast"] = reconciled.loc[mask, "ds"].map(brand_future * share).values

print(f"{len(cold_start_share)} cold-start series routed to the Brand-share fallback")
print(f"final_forecast differs from {mint_col} for {(reconciled['final_forecast'] != reconciled[mint_col]).sum()} rows")


319 cold-start series routed to the Brand-share fallback
final_forecast differs from lgbm/MinTrace_method-mint_shrink for 2544 rows
